### **Регрессия для CC50**

**Библиотеки**

In [1]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split, GridSearchCV
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score
)

**Загрузка данных**

In [3]:
X = pd.read_csv('./data/processed/X.csv', index_col=0)

X.head()

,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,NumValenceElectrons,MaxPartialCharge,MinPartialCharge,MaxAbsPartialCharge,...,fr_quatN,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea
0,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,158,0.038844,-0.293526,0.293526,...,0,0,0,0,0,0,0,0,3,0
1,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,162,0.012887,-0.313407,0.313407,...,0,0,0,0,0,0,0,0,3,0
2,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,186,0.094802,-0.325573,0.325573,...,2,0,0,0,0,0,0,0,3,0
3,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,164,0.038844,-0.293526,0.293526,...,0,0,0,0,0,0,0,0,4,0
4,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,184,0.062897,-0.257239,0.257239,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df_y = pd.read_csv('./data/processed/y_cc50.csv', index_col=0)
y = df_y['CC50_log']

y.head()

0    5.173221
1    1.856738
2    5.088474
3    4.690023
4    4.943576
Name: CC50_log, dtype: float64

**Разделение на обучающую и тестовую выборки**

In [5]:
print('Размерность X:', X.shape)
print('Размерность y:', y.shape)

Размерность X: (969, 178)
Размерность y: (969,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Размерность X_train:', X_train.shape)
print('Размерность y_train:', y_train.shape)

print('Размерность X_test:', X_test.shape)
print('Размерность y_test:', y_test.shape)

Размерность X_train: (775, 178)
Размерность y_train: (775,)
Размерность X_test: (194, 178)
Размерность y_test: (194,)


**Обучение моделей**

In [7]:
def evaluate_regression(y_true, y_pred, model_name):
    """Оценка регрессии: MAE, RMSE, R2"""
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(
        f'--- {model_name} ---\n'
        f'MAE:  {mae:.4f}\n'
        f'RMSE: {rmse:.4f}\n'
        f'R2:   {r2:.4f}'
    )
    
    return {
        'model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }


# Список для сохранения результатов моделей
model_results = []    

*LinearRegression (baseline)*

In [8]:
# Обучение LinearRegression с подбором параметров и кросс-валидацией

model = LinearRegression()

param_grid = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error'
)

grid.fit(X_train, y_train)

print('Лучшие параметры:', grid.best_params_)
print('Лучший RMSE:', -grid.best_score_)

Лучшие параметры: {'fit_intercept': True, 'positive': False}
Лучший RMSE: 1.4854084075064105


In [9]:
# Предсказание на лучшей LinearRegression модели и расчет метрик

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

model_results.append(
    evaluate_regression(y_test, y_pred, 'LinearRegression')
)

--- LinearRegression ---
MAE:  0.9693
RMSE: 1.3171
R2:   0.2615


*RandomForestRegressor*

In [10]:
# Обучение RandomForestRegressor с подбором параметров и кросс-валидацией

model = RandomForestRegressor(
    random_state=42, 
    n_jobs=-1
)

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [8, 10],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error'
)

grid.fit(X_train, y_train)

print('Лучшие параметры:', grid.best_params_)
print('Лучший RMSE:', -grid.best_score_)

Лучшие параметры: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Лучший RMSE: 1.203153108434842


In [11]:
# Предсказание на лучшей RandomForestRegressor модели и расчет метрик

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

model_results.append(
    evaluate_regression(y_test, y_pred, 'RandomForestRegressor')
)

--- RandomForestRegressor ---
MAE:  0.8781
RMSE: 1.2608
R2:   0.3233


*CatBoostRegressor*

In [12]:
# Обучение CatBoostRegressor с подбором параметров и кросс-валидацией

model = CatBoostRegressor(
    random_state=42, 
    verbose=0, 
    loss_function='RMSE'
)

param_grid = {
    'iterations': [300, 350],
    'depth': [6, 7],
    'learning_rate': [0.03, 0.05],
    'l2_leaf_reg': [3, 4]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('Лучшие параметры:', grid.best_params_)
print('Лучший RMSE:', -grid.best_score_)

Лучшие параметры: {'depth': 7, 'iterations': 350, 'l2_leaf_reg': 3, 'learning_rate': 0.03}
Лучший RMSE: 1.1929769434539912


In [13]:
# Предсказание на лучшей CatBoostRegressor модели и расчет метрик

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

model_results.append(
    evaluate_regression(y_test, y_pred, 'CatBoostRegressor')
)

--- CatBoostRegressor ---
MAE:  0.8698
RMSE: 1.2494
R2:   0.3355


*Сравнение моделей*

In [14]:
df_results = pd.DataFrame(model_results)

df_results

,model,MAE,RMSE,R2
0,LinearRegression,0.969333,1.317146,0.261489
1,RandomForestRegressor,0.878082,1.260840,0.323280
2,CatBoostRegressor,0.869845,1.249441,0.335461
